# D.R.O.N.A. — Notebook 08: Diffusion Policy ablation (LeRobot)

Trains a **Diffusion Policy** on the same gesture demonstrations as an ablation against ACT and the keyframe baseline (**C3**). Needs a **GPU**. ~20–40 min. Output: `data/checkpoints/diffusion/`.

You can run this right after notebook 07 in the same session (the dataset is reused).

## 1. Setup (edit `REPO_URL`)

In [ ]:
# ===========================================================================
#  D.R.O.N.A. — Colab / Kaggle setup.  *** RUN THIS CELL FIRST ***
#  Works on Google Colab and Kaggle. See docs/COLAB_TRAINING_GUIDE.md.
# ===========================================================================
import os, sys, subprocess, pathlib

# 1) GPU check ---------------------------------------------------------------
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "!!! NO GPU SELECTED.\n"
             "    Colab : Runtime > Change runtime type > T4 GPU\n"
             "    Kaggle: right panel > Settings > Accelerator > GPU T4 x2")

# 2) Get the D.R.O.N.A. repo -------------------------------------------------
#    EDIT REPO_URL to your GitHub repo. Private repo? use a read token:
#      https://<TOKEN>@github.com/<user>/D.R.O.N.A.git
#    OR leave it as-is and instead UPLOAD a zip of the project (Colab: Files panel)
#    / attach it as a Kaggle dataset named 'drona' — the search loop finds it.
REPO_URL = "https://github.com/<your-username>/D.R.O.N.A.git"   # <-- EDIT ME

def _is_repo(p):
    return pathlib.Path(p, "drona", "__init__.py").is_file()

search = ["D.R.O.N.A", ".", "/content/D.R.O.N.A",
          "/kaggle/working/D.R.O.N.A", "/kaggle/input/drona/D.R.O.N.A", "/kaggle/input/drona"]
repo = next((p for p in search if _is_repo(p)), None)
if repo is None and "<your-username>" not in REPO_URL:
    dest = "/content/D.R.O.N.A" if pathlib.Path("/content").is_dir() else "D.R.O.N.A"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, dest], check=True)
    repo = dest
assert repo and _is_repo(repo), (
    "Repo not found. Set REPO_URL to your GitHub URL, OR upload/attach the project, "
    "then re-run. See docs/COLAB_TRAINING_GUIDE.md.")
os.chdir(repo)
print("repo:", os.getcwd())

# 3) Install the drona package (light deps; training deps are installed below)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=False)
print("setup complete -- continue to the next cell")

## 2. Install LeRobot

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "git+https://github.com/huggingface/lerobot.git"], check=True)
print("lerobot installed")

## 3. Ensure the dataset exists (rebuild if needed)

In [ ]:
import os, pathlib
if not os.path.exists("data/lerobot/drona_gestures"):
    subprocess.run([sys.executable, "scripts/collect_demonstrations.py",
                    "--episodes", "25", "--out-dir", "data/demonstrations"], check=True)
    from drona.interaction.demonstration import DemonstrationDataset
    from drona.interaction.lerobot_dataset import build_lerobot_dataset
    demo = DemonstrationDataset.load_jsonl(pathlib.Path("data/demonstrations/demonstrations.jsonl"))
    build_lerobot_dataset(demo, repo_id="drona/gestures", fps=20, root="data/lerobot/drona_gestures")
print("dataset ready")

## 4. Train Diffusion Policy

In [ ]:
!python -m lerobot.scripts.train \
    --dataset.repo_id=drona/gestures \
    --dataset.root=data/lerobot/drona_gestures \
    --policy.type=diffusion \
    --policy.horizon=16 \
    --policy.n_action_steps=8 \
    --policy.n_obs_steps=2 \
    --batch_size=32 \
    --steps=30000 \
    --output_dir=data/checkpoints/diffusion \
    --job_name=drona_diffusion \
    --device=cuda \
    --wandb.enable=false

## 5. Three-way comparison: keyframe vs ACT vs Diffusion

In [ ]:
import json
from drona.interaction.act_policy import KeyframePolicy, LeRobotACTPolicy
from drona.interaction.diffusion_policy import LeRobotDiffusionPolicy
from drona.interaction.sim_eval import evaluate_policy

def factory(cls, base):
    def f(g):
        for path in (f"{base}/{g}", base):
            try:
                return cls(path, device="cuda")
            except Exception:
                continue
        return KeyframePolicy(g)
    return f

reports = {
    "keyframe":  evaluate_policy(lambda g: KeyframePolicy(g)),
    "act":       evaluate_policy(factory(LeRobotACTPolicy, "data/checkpoints/act")),
    "diffusion": evaluate_policy(factory(LeRobotDiffusionPolicy, "data/checkpoints/diffusion")),
}
for name, r in reports.items():
    print(f"{name:10} success={r.success_rate:.0%}  mean_jerk={r.mean_jerk:.4f}")

## 6. Download the trained Diffusion checkpoint

In [ ]:
# === Download your trained model back to your computer ===
# Edit SRC if you trained into a different directory.
import shutil, pathlib
SRC = "data/checkpoints/diffusion"
base = "/content" if pathlib.Path("/content").is_dir() else "/kaggle/working"
zip_path = shutil.make_archive(f"{base}/drona_diffusion_checkpoints", "zip", SRC)
print("zipped:", zip_path, "->", round(pathlib.Path(zip_path).stat().st_size/1e6, 1), "MB")
try:
    from google.colab import files
    files.download(zip_path)                      # Colab: browser download starts
except Exception:
    print("Kaggle: open the right-hand 'Output' tab, find the .zip, and click Download.")
print("\nNext (on your PC): unzip into the repo so the paths match:\n  unzip drona_diffusion_checkpoints.zip -d data/checkpoints/diffusion")